# Part 7 — Feature Engineering & the ML Reduction

**PyTimeSeries user guide** · *Turn a series into a supervised problem; wrap any regressor as a forecaster.*

> Uses the `pytimeseries` package on the shared commodity-price dataset.

In [1]:
import warnings; warnings.simplefilter("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import pytimeseries as pt
plt.rcParams.update({"figure.figsize": (10, 3.8), "axes.grid": True, "grid.alpha": .3,
                     "axes.spines.top": False, "axes.spines.right": False})

# Shared dataset: 356 months of global commodity prices (1992-2021)
raw = pd.read_csv("monthly_price.csv")
dates = pd.to_datetime(raw["YEAR"].astype(str) + "-" + raw["MONTH"], format="%Y-%b")
df = raw.drop(columns=["ID", "YEAR", "MONTH", "TIME"]); df.index = dates
df.index.name = "date"; df = df.asfreq("MS")
y = df["Food_Price_Index"]
print("pytimeseries", pt.__version__, "| series:", list(df.columns))

pytimeseries 0.5.0 | series: ['Urea_Price', 'MP_Price', 'TSP_Price', 'Maize_Price', 'Rice_Price', 'Wheat_Price', 'Food_Price_Index', 'Energy_Price_Index', 'VIX_Index']


In [2]:
# leakage-guarded lag matrix
X, target = pt.reduction.make_lag_matrix(y, [1, 2, 3, 12])
X.head(3)

,lag_1,lag_2,lag_3,lag_12
date,,,,
1993-01-01,52.0,52.1,51.7,55.7
1993-02-01,53.5,52.0,52.1,55.5
1993-03-01,53.3,53.5,52.0,55.6


In [3]:
# make_reduction turns a regressor into a forecaster (recursive or direct)
cv = pt.ExpandingWindow(h=12, n_splits=5)
from sklearn.linear_model import Ridge
rec = pt.make_reduction(Ridge(alpha=1.0), strategy='recursive', lags=12)
pt.evaluate(rec, y, cv=cv, metrics=('MASE',), sp=12).summary

metric,MASE
model,
Drift,0.7496
Naive,0.7579
RecursiveReductionForecaster,0.7949
SeasonalNaive(sp=12),0.9055


**Takeaway.** `make_lag_matrix` + `make_reduction` are the bridge from statistical models to the entire scikit-learn ecosystem — same contract, same backtest.